# Метод 4 — Set Transformer. Часть 2: обучение и ablation

| variant      | attention | distribution head | loss          |
|--------------|-----------|-------------------|---------------|
| A_full       | ISAB+PMA  | tail-cumsum       | smlar_smooth  |
| B_no_attn    | mean-pool | tail-cumsum       | smlar_smooth  |
| C_no_distr   | ISAB+PMA  | 3 sigmoid         | smlar_smooth  |
| D_no_smlar   | ISAB+PMA  | tail-cumsum       | MSE           |
| E_none       | mean-pool | 3 sigmoid         | MSE           |

In [1]:
from pathlib import Path
import sys

ROOT = Path('/Users/ensamsanovich/auc_forecast')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT_DIR = ROOT / 'outputs' / 'method_4_set_transformer'
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from core.config import N_PUBLISHERS, TARGETS
from core.leak_safe_features import UserHistoryIndex


CAMPAIGN_CONT_COLS = ["cpm", "hour_start", "duration", "audience_size", "n_publishers"]
N_USER_FEATURES = 9  


def precompute_campaign_features(val_df: pd.DataFrame) -> pd.DataFrame:
    df = val_df.copy()
    df["duration"] = df["hour_end"] - df["hour_start"]
    pubs = df["publishers"]
    if len(pubs) > 0 and isinstance(pubs.iloc[0], (list, tuple)):
        df["n_publishers"] = pubs.apply(len).astype(np.int32)
    else:
        df["n_publishers"] = pubs.astype(str).str.split(",").str.len().astype(np.int32)
    return df


def _as_user_id_array(value) -> np.ndarray:
    if isinstance(value, (list, tuple, np.ndarray)):
        return np.asarray(value, dtype=np.int32)
    return np.array([int(x) for x in str(value).split(",") if x], dtype=np.int32)


def _publisher_multihot(value, n_pubs: int = N_PUBLISHERS) -> np.ndarray:
    out = np.zeros(n_pubs, dtype=np.float32)
    iterable = value if isinstance(value, (list, tuple, np.ndarray)) else str(value).split(",")
    for p in iterable:
        idx = int(p) - 1  # publishers are 1-indexed in raw data
        if 0 <= idx < n_pubs:
            out[idx] = 1.0
    return out


@dataclass
class _DemoLookup:
    sex: np.ndarray
    age: np.ndarray
    city: np.ndarray
    known: np.ndarray
    max_uid: int

    @classmethod
    def build(cls, users_df: pd.DataFrame, age_max: int = 120, city_max: int = 3000) -> "_DemoLookup":
        u = users_df.set_index("user_id")
        max_uid = int(u.index.max())
        sex = np.zeros(max_uid + 2, dtype=np.float32)
        age = np.zeros(max_uid + 2, dtype=np.float32)
        city = np.zeros(max_uid + 2, dtype=np.float32)
        known = np.zeros(max_uid + 2, dtype=np.float32)
        for uid, r in u.iterrows():
            sex[uid] = float(r["sex"]) / 2.0
            age[uid] = float(r["age"]) / age_max
            city[uid] = float(r["city_id"]) / city_max
            known[uid] = 1.0
        return cls(sex=sex, age=age, city=city, known=known, max_uid=max_uid)

    def demo_block(self, user_ids: np.ndarray) -> np.ndarray:
        """[n_users, 4] -> (sex, age, city, is_unknown_user)."""
        demo = np.zeros((len(user_ids), 4), dtype=np.float32)
        for j, uid in enumerate(user_ids):
            if 0 <= uid <= self.max_uid and self.known[uid] > 0:
                demo[j, 0] = self.sex[uid]
                demo[j, 1] = self.age[uid]
                demo[j, 2] = self.city[uid]
            else:
                demo[j, 3] = 1.0
        return demo


class CampaignSetDataset(Dataset):

    def __init__(
        self,
        campaign_rows: pd.DataFrame,
        users_df: pd.DataFrame,
        hist_index: UserHistoryIndex,
        targets_df: pd.DataFrame | None,
        camp_feat_mean: np.ndarray,
        camp_feat_std: np.ndarray,
        user_feat_mean: np.ndarray,
        user_feat_std: np.ndarray,
        n_publishers: int = N_PUBLISHERS,
    ):
        self.campaigns = campaign_rows.reset_index(drop=True)
        self.targets_df = targets_df.reset_index(drop=True) if targets_df is not None else None
        self.n_publishers = n_publishers
        demo = _DemoLookup.build(users_df)

        self._user_feat_cache: list[torch.Tensor] = []
        self._camp_feat_cache: list[torch.Tensor] = []
        self._pub_cache: list[torch.Tensor] = []
        self._target_cache: list[torch.Tensor] = []
        self._n_users_cache: list[int] = []

        for idx in range(len(self.campaigns)):
            row = self.campaigns.iloc[idx]
            cutoff = int(row["hour_start"])
            user_ids = _as_user_id_array(row["user_ids"])

            hist_feats = np.empty((len(user_ids), 5), dtype=np.float32)
            for j, uid in enumerate(user_ids):
                hist_feats[j] = hist_index.user_features_before(int(uid), cutoff)
            hist_std = (hist_feats[:, :4] - user_feat_mean[:4]) / (user_feat_std[:4] + 1e-6)

            demo_block = demo.demo_block(user_ids)
            user_feats = np.concatenate(
                [hist_std, hist_feats[:, 4:5], demo_block], axis=1
            ).astype(np.float32)
            assert user_feats.shape[1] == N_USER_FEATURES

            camp_raw = np.array([row[c] for c in CAMPAIGN_CONT_COLS], dtype=np.float32)
            camp_feats = ((camp_raw - camp_feat_mean) / (camp_feat_std + 1e-6)).astype(np.float32)
            pub_mh = _publisher_multihot(row["publishers"], n_publishers)

            self._user_feat_cache.append(torch.from_numpy(user_feats))
            self._camp_feat_cache.append(torch.from_numpy(camp_feats))
            self._pub_cache.append(torch.from_numpy(pub_mh))
            self._n_users_cache.append(int(len(user_ids)))
            if self.targets_df is not None:
                self._target_cache.append(torch.tensor(
                    [self.targets_df.loc[idx, t] for t in TARGETS], dtype=torch.float32
                ))

    def __len__(self) -> int:
        return len(self.campaigns)

    def __getitem__(self, idx: int) -> dict:
        out = {
            "user_feats": self._user_feat_cache[idx],
            "camp_feats": self._camp_feat_cache[idx],
            "pub_mh": self._pub_cache[idx],
            "n_users": self._n_users_cache[idx],
        }
        if self.targets_df is not None:
            out["target"] = self._target_cache[idx]
        return out


def collate_pad(batch: list[dict]) -> dict:
    max_n = max(b["n_users"] for b in batch)
    B = len(batch)
    D_user = batch[0]["user_feats"].shape[1]
    user_feats = torch.zeros(B, max_n, D_user)
    mask = torch.zeros(B, max_n, dtype=torch.bool)
    camp_feats = torch.stack([b["camp_feats"] for b in batch])
    pub_mh = torch.stack([b["pub_mh"] for b in batch])
    has_target = "target" in batch[0]
    targets = torch.stack([b["target"] for b in batch]) if has_target else None
    for i, b in enumerate(batch):
        n = b["n_users"]
        user_feats[i, :n] = b["user_feats"]
        mask[i, :n] = True
    return {
        "user_feats": user_feats,
        "mask": mask,
        "camp_feats": camp_feats,
        "pub_mh": pub_mh,
        "target": targets,
    }


def fit_normalizers(
    campaigns: pd.DataFrame,
    train_idx: np.ndarray,
    hist_index: UserHistoryIndex,
    sample_seed: int = 42,
    user_sample_cap: int = 200,
) -> dict:
    train_camps = campaigns.iloc[train_idx]
    camp_X = train_camps[CAMPAIGN_CONT_COLS].to_numpy(dtype=np.float32)
    camp_mean = camp_X.mean(axis=0)
    camp_std = camp_X.std(axis=0) + 1e-6

    rng = np.random.default_rng(sample_seed)
    user_feat_samples = []
    for _, row in train_camps.iterrows():
        cutoff = int(row["hour_start"])
        uids = _as_user_id_array(row["user_ids"])
        if len(uids) > user_sample_cap:
            uids = rng.choice(uids, user_sample_cap, replace=False)
        for uid in uids:
            user_feat_samples.append(hist_index.user_features_before(int(uid), cutoff))
    U = np.stack(user_feat_samples, axis=0)
    user_mean = U[:, :4].mean(axis=0)
    user_std = U[:, :4].std(axis=0) + 1e-6
    return {
        "camp_mean": camp_mean,
        "camp_std": camp_std,
        "user_mean": user_mean,
        "user_std": user_std,
    }


In [ ]:
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class MAB(nn.Module):

    def __init__(self, d_q: int, d_k: int, d_out: int, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert d_out % n_heads == 0, f"d_out ({d_out}) must be divisible by n_heads ({n_heads})"
        self.n_heads = n_heads
        self.d_head = d_out // n_heads
        self.fc_q = nn.Linear(d_q, d_out)
        self.fc_k = nn.Linear(d_k, d_out)
        self.fc_v = nn.Linear(d_k, d_out)
        self.fc_o = nn.Linear(d_out, d_out)
        self.ln0 = nn.LayerNorm(d_out)
        self.ln1 = nn.LayerNorm(d_out)
        self.ff = nn.Sequential(
            nn.Linear(d_out, d_out * 2), nn.GELU(), nn.Linear(d_out * 2, d_out)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q: torch.Tensor, K: torch.Tensor, mask_k: torch.Tensor | None = None) -> torch.Tensor:
        B, m, _ = Q.shape
        n = K.shape[1]
        q = self.fc_q(Q).view(B, m, self.n_heads, self.d_head).transpose(1, 2)
        k = self.fc_k(K).view(B, n, self.n_heads, self.d_head).transpose(1, 2)
        v = self.fc_v(K).view(B, n, self.n_heads, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask_k is not None:
            scores = scores.masked_fill(~mask_k[:, None, None, :], float("-inf"))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, m, self.n_heads * self.d_head)
        out = self.fc_o(out)
        # Project Q via fc_q for residual (so dims match) — equivalent to taking the residual on `q` reshaped.
        q_skip = self.fc_q(Q) if Q.shape[-1] != out.shape[-1] else Q
        h = self.ln0(q_skip + out)
        h = self.ln1(h + self.ff(h))
        return h


class ISAB(nn.Module):
    def __init__(self, d_in: int, d_out: int, m_ind: int = 16, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.inducing = nn.Parameter(torch.randn(1, m_ind, d_out) * 0.02)
        self.mab1 = MAB(d_out, d_in, d_out, n_heads, dropout)
        self.mab2 = MAB(d_in, d_out, d_out, n_heads, dropout)

    def forward(self, X: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        B = X.shape[0]
        I = self.inducing.expand(B, -1, -1)
        H = self.mab1(I, X, mask_k=mask)          # [B, m_ind, d_out]
        out = self.mab2(X, H, mask_k=None)         # [B, n, d_out]
        return out


class PMA(nn.Module):

    def __init__(self, d: int, k: int = 1, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.seeds = nn.Parameter(torch.randn(1, k, d) * 0.02)
        self.mab = MAB(d, d, d, n_heads, dropout)

    def forward(self, X: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        B = X.shape[0]
        S = self.seeds.expand(B, -1, -1)
        return self.mab(S, X, mask_k=mask)         # [B, k, d]


class SetTransformerReach(nn.Module):
    def __init__(
        self,
        d_user: int,
        d_camp: int,
        n_publishers: int = 21,
        d_hidden: int = 96,
        n_isab: int = 2,
        m_ind: int = 16,
        n_heads: int = 4,
        k_max: int = 6,
        dropout: float = 0.1,
        use_attention: bool = True,
        use_distribution: bool = True,
    ):
        super().__init__()
        self.use_attention = use_attention
        self.use_distribution = use_distribution
        self.k_max = k_max
        self.d_hidden = d_hidden

        self.user_enc = nn.Sequential(
            nn.Linear(d_user, d_hidden),
            nn.GELU(),
            nn.Linear(d_hidden, d_hidden),
        )

        if use_attention:
            self.isab = nn.ModuleList(
                [ISAB(d_hidden, d_hidden, m_ind, n_heads, dropout) for _ in range(n_isab)]
            )
            self.pma = PMA(d_hidden, k=1, n_heads=n_heads, dropout=dropout)
        else:
            self.isab = None
            self.pma = None

        self.camp_enc = nn.Sequential(
            nn.Linear(d_camp + n_publishers, d_hidden),
            nn.GELU(),
            nn.Linear(d_hidden, d_hidden),
        )

        head_in = d_hidden * 2
        head_out = (k_max + 1) if use_distribution else 3
        self.head = nn.Sequential(
            nn.Linear(head_in, d_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, head_out),
        )

    def forward(self, batch: dict) -> dict:
        X = batch["user_feats"]               # [B, n, D_user]
        mask = batch["mask"]                  # [B, n]
        camp = batch["camp_feats"]            # [B, D_camp]
        pub = batch["pub_mh"]                 # [B, n_pubs]

        H = self.user_enc(X)                  # [B, n, d_hidden]
        if self.use_attention:
            for blk in self.isab:
                H = blk(H, mask=mask)
            pooled = self.pma(H, mask=mask).squeeze(1)        # [B, d_hidden]
        else:
            m = mask.unsqueeze(-1).float()
            pooled = (H * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)

        camp_repr = self.camp_enc(torch.cat([camp, pub], dim=1))
        joint = torch.cat([pooled, camp_repr], dim=1)
        logits = self.head(joint)

        if self.use_distribution:
            probs = F.softmax(logits, dim=1)                  # [B, k_max+1]
            # tail-cumsum: y_k = P(X >= k) = sum_{m=k..K} probs[m]
            cum_from_top = torch.cumsum(probs.flip(1), dim=1).flip(1)  # [B, k_max+1]
            y_hat = cum_from_top[:, 1:4]                       # [B, 3]
            return {"y_hat": y_hat, "probs": probs}
        else:
            y_hat = torch.sigmoid(logits)                      # ablation: no monotone guarantee
            return {"y_hat": y_hat, "probs": None}


In [ ]:
from __future__ import annotations

import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from core.config import N_PUBLISHERS, SEED, TARGETS, set_seed
from core.leak_safe_features import UserHistoryIndex
from core.metrics import (
    monotonicity_violation,
    smlar,
    smlar_per_target,
    smlar_smooth_loss,
    mse_loss,
    mae_loss,
)
from core.splits import build_unified_split
_LOSSES = {
    "smlar_smooth": smlar_smooth_loss,
    "mse": mse_loss,
    "mae": mae_loss,
}


def _get_loss(name: str) -> torch.nn.Module:
    if name not in _LOSSES:
        raise ValueError(f"Unknown loss {name!r}. Available: {sorted(_LOSSES)}")
    return _LOSSES[name]()

def get_device() -> torch.device:
    override = os.environ.get("AUC_TORCH_DEVICE", "").lower()
    if override in {"cuda", "cuda:0"}:
        if not torch.cuda.is_available():
            raise RuntimeError("AUC_TORCH_DEVICE=cuda but CUDA is not available.")
        return torch.device("cuda")
    if override == "mps":
        if not (hasattr(torch.backends, "mps") and torch.backends.mps.is_available()):
            raise RuntimeError("AUC_TORCH_DEVICE=mps but MPS is not available.")
        return torch.device("mps")
    if override == "cpu":
        return torch.device("cpu")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


BASE_CONFIG: dict = {
    "batch_size": 16,
    "lr": 1e-3,
    "wd": 1e-4,
    "epochs": 50,
    "d_hidden": 96,
    "n_isab": 2,
    "m_ind": 16,
    "n_heads": 4,
    "k_max": 5,
    "dropout": 0.1,
}
VARIANTS: list[tuple[str, dict]] = [
    ("A_full",      {"use_attention": True,  "use_distribution": True,  "loss": "smlar_smooth"}),
    ("B_no_attn",   {"use_attention": False, "use_distribution": True,  "loss": "smlar_smooth"}),
    ("C_no_distr",  {"use_attention": True,  "use_distribution": False, "loss": "smlar_smooth"}),
    ("D_no_smlar",  {"use_attention": True,  "use_distribution": True,  "loss": "mse"}),
    ("E_none",      {"use_attention": False, "use_distribution": False, "loss": "mse"}),
]

def _build_model(config: dict) -> SetTransformerReach:
    return SetTransformerReach(
        d_user=N_USER_FEATURES,
        d_camp=len(CAMPAIGN_CONT_COLS),
        n_publishers=N_PUBLISHERS,
        d_hidden=config["d_hidden"],
        n_isab=config["n_isab"],
        m_ind=config["m_ind"],
        n_heads=config["n_heads"],
        k_max=config["k_max"],
        dropout=config["dropout"],
        use_attention=config["use_attention"],
        use_distribution=config["use_distribution"],
    )


def train_one_fold(
    campaigns: pd.DataFrame,
    targets: pd.DataFrame,
    users_df: pd.DataFrame,
    hist_index: UserHistoryIndex,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    *,
    config: dict,
    seed: int = SEED,
    verbose: bool = False,
    save_checkpoint: Path | None = None,
) -> tuple[dict, SetTransformerReach, dict]:
    """Train one fold; return (best metrics + arrays, trained model, normalizers)."""
    set_seed(seed)
    device = get_device()

    norm = fit_normalizers(campaigns, train_idx, hist_index, sample_seed=seed)

    train_ds = CampaignSetDataset(
        campaigns.iloc[train_idx], users_df, hist_index, targets.iloc[train_idx],
        norm["camp_mean"], norm["camp_std"], norm["user_mean"], norm["user_std"],
    )
    val_ds = CampaignSetDataset(
        campaigns.iloc[val_idx], users_df, hist_index, targets.iloc[val_idx],
        norm["camp_mean"], norm["camp_std"], norm["user_mean"], norm["user_std"],
    )

    train_loader = DataLoader(
        train_ds, batch_size=config["batch_size"], shuffle=True,
        collate_fn=collate_pad, num_workers=0, drop_last=False,
    )
    val_loader = DataLoader(
        val_ds, batch_size=config["batch_size"], shuffle=False,
        collate_fn=collate_pad, num_workers=0,
    )

    model = _build_model(config).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=config["epochs"])
    loss_fn = _get_loss(config["loss"]).to(device)

    best: dict = {"smlar": float("inf"), "epoch": -1, "pred": None, "true": None, "state_dict": None}
    for ep in range(config["epochs"]):
        model.train()
        train_losses = []
        for batch in train_loader:
            batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
            opt.zero_grad()
            out = model(batch)
            loss = loss_fn(out["y_hat"], batch["target"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            train_losses.append(loss.item())
        sched.step()

        model.eval()
        all_pred, all_true = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
                out = model(batch)
                all_pred.append(out["y_hat"].detach().cpu().numpy())
                all_true.append(batch["target"].detach().cpu().numpy())
        y_pred = np.clip(np.concatenate(all_pred, axis=0), 0.0, 1.0)
        y_true = np.concatenate(all_true, axis=0)
        sm = smlar(y_true, y_pred)
        if sm < best["smlar"]:
            best = {
                "smlar": sm,
                "epoch": ep,
                "pred": y_pred,
                "true": y_true,
                "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            }
        if verbose and (ep % 5 == 0 or ep == config["epochs"] - 1):
            print(
                f"    ep {ep:3d}  trainL={np.mean(train_losses):.4f}  "
                f"valSMLAR={sm:.2f}%  best={best['smlar']:.2f}%"
            )

    if save_checkpoint is not None and best["state_dict"] is not None:
        save_checkpoint = Path(save_checkpoint)
        save_checkpoint.parent.mkdir(parents=True, exist_ok=True)
        torch.save({"state_dict": best["state_dict"], "config": config, "norm": norm}, save_checkpoint)

    return best, model, norm

def run_cv(
    config: dict,
    *,
    data: dict | None = None,
    n_folds: int = 5,
    seed: int = SEED,
    verbose: bool = False,
) -> tuple[pd.DataFrame, dict, np.ndarray, np.ndarray]:
    """Run 5-fold CV on the unified split.

    `data` should be the dict returned by `core.runner.load_unified_data`. If None,
    it is loaded here (slower if called multiple times — pre-load once and pass in).
    """
    if data is None:
        from core.runner import load_unified_data
        data = load_unified_data()
    campaigns = precompute_campaign_features(data["val"])
    hist_index = data["user_index"]
    users_df = data["users"]
    ans = data["ans"]
    split = data["split"] if "split" in data else build_unified_split(data["val"])

    fold_results = []
    for fold_i, (tr, va) in enumerate(split["folds"]):
        t0 = time.time()
        best, _, _ = train_one_fold(
            campaigns, ans, users_df, hist_index, tr, va,
            config=config, seed=seed + fold_i, verbose=verbose,
        )
        elapsed = time.time() - t0
        per_t = smlar_per_target(best["true"], best["pred"])
        mono = monotonicity_violation(best["pred"])
        fold_results.append({
            "fold": fold_i,
            "smlar": best["smlar"],
            "best_epoch": best["epoch"],
            "smlar_y1": per_t["at_least_one"],
            "smlar_y2": per_t["at_least_two"],
            "smlar_y3": per_t["at_least_three"],
            "monotone_violation": mono,
            "time_s": elapsed,
        })
        if verbose:
            print(
                f"  fold {fold_i} best epoch {best['epoch']}  "
                f"SMLAR={best['smlar']:.2f}%  ({elapsed:.1f}s)"
            )

    fold_df = pd.DataFrame(fold_results)
    summary = {
        "mean_smlar": float(fold_df["smlar"].mean()),
        "std_smlar": float(fold_df["smlar"].std()),
        "mean_mono_violation": float(fold_df["monotone_violation"].mean()),
        "config": config,
    }
    return fold_df, summary, split["train_idx"], split["holdout_idx"]


def run_holdout(
    config: dict,
    *,
    data: dict | None = None,
    seed: int = SEED,
    verbose: bool = False,
    save_checkpoint: Path | None = None,
) -> dict:
    if data is None:
        from core.runner import load_unified_data
        data = load_unified_data()
    campaigns = precompute_campaign_features(data["val"])
    hist_index = data["user_index"]
    users_df = data["users"]
    ans = data["ans"]
    split = data["split"] if "split" in data else build_unified_split(data["val"])

    best, _, _ = train_one_fold(
        campaigns, ans, users_df, hist_index,
        split["train_idx"], split["holdout_idx"],
        config=config, seed=seed, verbose=verbose,
        save_checkpoint=save_checkpoint,
    )
    per_t = smlar_per_target(best["true"], best["pred"])
    return {
        "holdout_smlar": best["smlar"],
        "best_epoch": best["epoch"],
        "smlar_y1": per_t["at_least_one"],
        "smlar_y2": per_t["at_least_two"],
        "smlar_y3": per_t["at_least_three"],
        "monotone_violation": monotonicity_violation(best["pred"]),
        "pred": best["pred"],
        "true": best["true"],
    }


In [9]:
from core.runner import load_unified_data
from core.splits import build_unified_split

data = load_unified_data()
campaigns = precompute_campaign_features(data['val'])
split = data['split']

config = dict(BASE_CONFIG, epochs=2, **dict(VARIANTS[0][1]))
tr, va = split['folds'][0]
best, _, _ = train_one_fold(
    campaigns, data['ans'], data['users'], data['user_index'], tr, va,
    config=config, verbose=True,
)
print('Smoke best SMLAR:', best['smlar'])

    ep   0  trainL=1.3655  valSMLAR=175.91%  best=175.91%
    ep   1  trainL=0.8266  valSMLAR=143.15%  best=143.15%
Smoke best SMLAR: 143.1469746592933


In [12]:

import pandas as pd
rows = []
for name, overrides in VARIANTS:
    cfg = dict(BASE_CONFIG, **overrides)
    fold_df, summary, *_ = run_cv(cfg, data=data, verbose=True)
    rows.append({
        'variant': name,
        'use_attention': cfg['use_attention'],
        'use_distribution': cfg['use_distribution'],
        'loss': cfg['loss'],
        'cv_smlar_mean': summary['mean_smlar'],
        'cv_smlar_std': summary['std_smlar'],
        'monotone_violation': summary['mean_mono_violation'],
    })
ablation = pd.DataFrame(rows).sort_values('cv_smlar_mean')
ablation.to_csv(Path(OUT_DIR) / 'ablation_summary.csv', index=False)
ablation

    ep   0  trainL=1.3655  valSMLAR=175.91%  best=175.91%
    ep   5  trainL=0.4542  valSMLAR=64.04%  best=64.04%
    ep  10  trainL=0.3243  valSMLAR=55.22%  best=51.41%
    ep  15  trainL=0.2831  valSMLAR=46.36%  best=40.98%
    ep  20  trainL=0.2631  valSMLAR=36.92%  best=36.92%
    ep  25  trainL=0.2192  valSMLAR=37.07%  best=34.77%
    ep  30  trainL=0.2220  valSMLAR=33.96%  best=33.96%
    ep  35  trainL=0.1989  valSMLAR=32.78%  best=31.94%
    ep  40  trainL=0.1907  valSMLAR=31.31%  best=30.79%
    ep  45  trainL=0.1845  valSMLAR=30.96%  best=30.78%
    ep  49  trainL=0.1827  valSMLAR=30.97%  best=30.78%
  fold 0 best epoch 44  SMLAR=30.78%  (218.8s)
    ep   0  trainL=1.4582  valSMLAR=499.82%  best=499.82%
    ep   5  trainL=0.5709  valSMLAR=80.15%  best=80.15%
    ep  10  trainL=0.4569  valSMLAR=60.78%  best=60.78%
    ep  15  trainL=0.3626  valSMLAR=40.23%  best=40.23%
    ep  20  trainL=0.2985  valSMLAR=34.39%  best=34.39%
    ep  25  trainL=0.2529  valSMLAR=27.59%  best=27.5

,variant,use_attention,use_distribution,loss,cv_smlar_mean,cv_smlar_std,monotone_violation
0,A_full,True,True,smlar_smooth,26.596144,2.583674,0.000000
2,C_no_distr,True,False,smlar_smooth,26.878755,2.190975,0.000000
1,B_no_attn,False,True,smlar_smooth,28.856842,2.120833,0.000000
4,E_none,False,False,mse,61.243256,3.070702,0.003727
3,D_no_smlar,True,True,mse,67.556748,11.660804,0.000000
